# 🧠 Notebook 17: Alignment — RLHF, DPO, and GRPO

How base language models become helpful, harmless, and honest — the math and methods behind modern alignment.

**Prerequisites:** Notebooks 07 (Building GPT), 08 (Training on Apple Silicon)

**What you'll learn:**
- 💡 Why alignment matters: base models are capable but not helpful or safe
- 🎯 The full alignment pipeline: Pretraining → SFT → RLHF → DPO → KTO → GRPO
- ⚡ Mathematical derivations for each transition, from reward modeling to direct preference optimization
- ⚠️ The tradeoffs: each method eliminates a bottleneck but introduces new assumptions

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

## Why Alignment Matters

A base language model trained on internet text is a powerful next-token predictor. But prediction ≠ helpfulness. Consider what a base model *actually* learns:

- **Capability without direction.** It can write poetry, code, and legal briefs — but also toxic content, misinformation, and harmful instructions. It has no preference for one over the other.
- **Mimicry, not intent.** The training objective (minimize cross-entropy on web text) rewards predicting what humans *wrote*, not what humans *want*. A model that perfectly predicts the internet would reproduce all its flaws.
- **The helpful assistant problem.** When you ask a base model "How do I fix this bug?", it might continue the text as if it were a forum post, a sarcastic reply, or a completely unrelated tangent. It doesn't *know* it should be helpful.

💡 **The core insight:** Capability and alignment are orthogonal axes. A model can be highly capable but poorly aligned (dangerous), or well-aligned but not very capable (useless). We want both.

⚠️ **The alignment tax:** Every alignment method constrains the model's behavior, potentially reducing raw capability. The art is minimizing this tax while maximizing helpfulness and safety.

## The Alignment Pipeline: Pretraining → SFT → Alignment

Modern LLM development follows a three-stage pipeline. Each stage builds on the previous one:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    THE ALIGNMENT PIPELINE                               │
│                                                                         │
│  Stage 1: PRETRAINING          Stage 2: SFT           Stage 3: ALIGNMENT│
│  ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐  │
│  │ Next-token pred  │───▶│ Instruction      │───▶│ RLHF / DPO /    │  │
│  │ on web text      │    │ following on      │    │ KTO / GRPO      │  │
│  │                  │    │ curated examples  │    │                  │  │
│  │ Loss: CE on      │    │ Loss: CE on       │    │ Loss: preference │  │
│  │ internet data    │    │ (instruction,     │    │ optimization     │  │
│  │                  │    │  response) pairs  │    │                  │  │
│  └──────────────────┘    └──────────────────┘    └──────────────────┘  │
│                                                                         │
│  Learns: language       Learns: format &         Learns: which          │
│  structure, facts,      instruction following    responses humans       │
│  reasoning patterns                              actually prefer        │
└─────────────────────────────────────────────────────────────────────────┘
```

### Stage 1: Pretraining (what we built in Notebooks 07–08)

Standard language modeling on trillions of tokens. The model learns grammar, facts, reasoning patterns, and unfortunately, all the biases and toxicity in the training data.

### Stage 2: Supervised Fine-Tuning (SFT)

Fine-tune on curated (instruction, response) pairs. This teaches the model the *format* of being helpful — it learns to respond to questions rather than continue text. But SFT alone can't teach *quality*: the model doesn't learn which of two valid responses is better.

$$\mathcal{L}_{\text{SFT}} = -\sum_{t=1}^{T} \log \pi_\theta(y_t \mid x, y_{<t})$$

where $x$ is the instruction and $y$ is the demonstration response.

### Stage 3: Alignment (this notebook)

This is where the magic happens. We teach the model to prefer *better* responses over *worse* ones, using human preferences rather than demonstrations. This is the focus of this notebook.

🎯 **Interview tip:** Be ready to explain why SFT alone isn't enough. The key insight is that SFT teaches format but not quality — it can't distinguish between a mediocre response and an excellent one if both follow the correct format.

## 🔢 Math Foundations: What You Need for This Notebook

Before we dive into alignment methods, let's build up three math tools we'll use repeatedly.

### Tool 1: The Sigmoid Function σ(x)

The sigmoid function squashes any number into the range (0, 1) — perfect for representing probabilities:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**Intuition — think of it as a "confidence meter":**
- σ(0) = 0.5 → "I'm 50/50, no idea"
- σ(2) = 0.88 → "Pretty confident yes"
- σ(-2) = 0.12 → "Pretty confident no"
- σ(10) ≈ 1.0 → "Absolutely yes"

### Tool 2: Log Rules We'll Use

We'll manipulate logarithms frequently. Three rules to remember:
- log(a/b) = log(a) - log(b)  ← "log of a ratio = difference of logs"
- log(a × b) = log(a) + log(b)
- exp(log(a)) = a  ← "exp undoes log"

### Tool 3: KL Divergence (Measuring Distribution Difference)

KL divergence measures how different two probability distributions are:

$$\text{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

**Analogy:** Imagine you have two bags of colored marbles. KL divergence measures how "surprised" you'd be if you thought you were drawing from bag Q but were actually drawing from bag P. If the bags are identical, KL = 0 (no surprise). The more different they are, the higher the KL.

**In alignment:** We use KL to measure how far our fine-tuned model has drifted from the original. Too much drift = the model might be "gaming" the reward signal.

---

## Math Derivation: RLHF (Reinforcement Learning from Human Feedback)

Following the pedagogical pattern: *math derivation → step-by-step code → visualization → comparison.*

### The Reward Model

The first ingredient of RLHF is a **reward model** $r_\phi(x, y)$ that assigns a scalar quality score to a (prompt, response) pair. This model is trained on human preference data.

Given a prompt $x$ and two responses $y_w$ (preferred/chosen) and $y_l$ (rejected), humans label which is better. The reward model is trained via the **Bradley-Terry model**:

$$P(y_w \succ y_l \mid x) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big)$$

where $\sigma$ is the sigmoid function. The training loss is:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \Big[\log \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big)\Big]$$

💡 **Key insight:** The reward model only needs to produce *relative* scores — the absolute scale doesn't matter. What matters is that $r(x, y_w) > r(x, y_l)$ for preferred responses.

### Architecture

In practice, the reward model is typically a copy of the SFT model with the language modeling head replaced by a linear projection to a scalar:

$$r_\phi(x, y) = W_r^\top \cdot h_{\text{last}}(x, y) + b_r$$

where $h_{\text{last}}$ is the last hidden state of the transformer at the final token position, and $W_r \in \mathbb{R}^{d_{\text{model}}}$, $b_r \in \mathbb{R}$.

### The RLHF Objective

With a trained reward model in hand, we optimize the policy (language model) $\pi_\theta$ to maximize expected reward. But there's a critical constraint: we can't let the policy drift too far from the reference model $\pi_{\text{ref}}$ (typically the SFT model), or it will find degenerate ways to hack the reward model.

The RLHF objective is:

$$\boxed{\max_{\pi_\theta} \; \mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_\theta(\cdot|x)} \Big[ r_\phi(x, y) \Big] - \beta \cdot \text{KL}\big(\pi_\theta \| \pi_{\text{ref}}\big)}$$

where:
- $r_\phi(x, y)$ is the learned reward model score
- $\beta > 0$ is the KL penalty coefficient (controls the alignment-capability tradeoff)
- $\text{KL}(\pi_\theta \| \pi_{\text{ref}}) = \mathbb{E}_{y \sim \pi_\theta} \left[\log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}\right]$ measures how far the policy has drifted

### Why the KL Penalty is Essential

Without the KL term, the policy would **reward hack** — finding adversarial outputs that score high on the (imperfect) reward model but are actually low quality:

⚠️ **Reward hacking examples:**
- Repeating phrases the reward model likes ("I'm happy to help! I'm happy to help!")
- Generating extremely long responses (if the reward model has a length bias)
- Producing outputs that exploit specific patterns in the reward model's training data

The KL penalty keeps the policy close to the SFT model, which acts as an anchor to sensible behavior. Higher $\beta$ means more conservative updates (less reward hacking risk, but slower alignment). Lower $\beta$ means more aggressive optimization (faster alignment, but higher risk of degenerate behavior).

### Expanding the Objective

Writing out the KL divergence explicitly:

$$\max_{\pi_\theta} \; \mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_\theta(\cdot|x)} \left[ r_\phi(x, y) - \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} \right]$$

This can be rewritten as maximizing a **modified reward**:

$$\tilde{r}(x, y) = r_\phi(x, y) - \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$

🎯 **Interview tip:** The RLHF objective is a constrained optimization problem. The reward term pulls the policy toward high-reward outputs, while the KL term pulls it back toward the reference. The balance point $\beta$ is one of the most important hyperparameters in alignment.

### The Optimal Policy (Closed-Form Solution)

A remarkable result: the RLHF objective has a **closed-form optimal policy**. Setting the gradient to zero and solving:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\left(\frac{1}{\beta} r_\phi(x, y)\right)$$

where $Z(x) = \sum_y \pi_{\text{ref}}(y|x) \exp\left(\frac{1}{\beta} r_\phi(x, y)\right)$ is the partition function (intractable to compute).

**Derivation sketch:**

1. Write the Lagrangian with the constraint that $\pi_\theta$ is a valid distribution
2. Take the functional derivative with respect to $\pi_\theta(y|x)$
3. Set to zero and solve for $\pi_\theta$

The result tells us: the optimal policy is the reference policy reweighted by exponentiated reward. High-reward responses get upweighted, low-reward responses get downweighted, all relative to the reference.

💡 **Why this matters:** This closed-form solution is the mathematical bridge to DPO. We can't compute $Z(x)$ directly, but we can *rearrange* this equation to eliminate the partition function entirely.

### RLHF in Practice: PPO

Since we can't compute the optimal policy directly (the partition function $Z(x)$ requires summing over all possible responses), RLHF uses **Proximal Policy Optimization (PPO)** to approximately solve the objective:

1. Sample responses $y \sim \pi_\theta(\cdot|x)$ from the current policy
2. Score them with the reward model: $r_\phi(x, y)$
3. Compute the policy gradient with KL penalty
4. Update $\pi_\theta$ with PPO's clipped objective

⚠️ **The RLHF bottleneck:** This requires:
- A separate trained reward model (expensive to train and maintain)
- Online sampling from the policy during training (slow)
- PPO optimization (notoriously unstable, sensitive to hyperparameters)
- Four models in memory simultaneously: policy, reference, reward model, value function

This complexity motivated the search for simpler alternatives — enter DPO.

---

## Math Derivation: DPO (Direct Preference Optimization)

DPO's key insight is elegant: **we can reparameterize the reward function in terms of the policy itself**, eliminating the need for a separate reward model entirely.

### Step 1: Reparameterize the Reward

Starting from the optimal policy under RLHF:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\left(\frac{1}{\beta} r(x, y)\right)$$

Take the log of both sides:

$$\log \pi^*(y|x) = \log \pi_{\text{ref}}(y|x) + \frac{1}{\beta} r(x, y) - \log Z(x)$$

Now solve for $r(x, y)$:

$$\boxed{r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)}$$

💡 **The key insight:** The reward is fully determined by the ratio of the optimal policy to the reference policy (plus a prompt-dependent constant). We don't need a separate reward model — the policy *is* the reward model.

### Step 2: Substitute into the Bradley-Terry Model

Recall the preference probability under Bradley-Terry:

$$P(y_w \succ y_l | x) = \sigma\big(r(x, y_w) - r(x, y_l)\big)$$

Substituting our reparameterized reward:

$$P(y_w \succ y_l | x) = \sigma\left(\beta \log \frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} + \cancel{\beta \log Z(x)} - \beta \log \frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)} - \cancel{\beta \log Z(x)}\right)$$

⚡ **The partition function cancels!** This is the magic of DPO — the intractable $Z(x)$ drops out because it appears in both the chosen and rejected reward terms.

### Step 3: The DPO Loss

Replacing $\pi^*$ with our trainable policy $\pi_\theta$ and taking the negative log-likelihood:

$$\boxed{\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right)\right]}$$

Or more compactly, defining the **log-ratio** for a response $y$:

$$\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$

The loss becomes:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\big(\hat{r}_\theta(x, y_w) - \hat{r}_\theta(x, y_l)\big)\right]$$

This is just a binary cross-entropy loss on the implicit reward margin between chosen and rejected responses.

### Why DPO Eliminates the Reward Model

Let's be precise about what DPO removes from the RLHF pipeline:

```
RLHF Pipeline (4 models in memory):          DPO Pipeline (2 models in memory):
┌──────────────┐                              ┌──────────────┐
│ Policy πθ    │ ← being optimized            │ Policy πθ    │ ← being optimized
├──────────────┤                              ├──────────────┤
│ Reference πref│ ← frozen copy               │ Reference πref│ ← frozen copy
├──────────────┤                              └──────────────┘
│ Reward Model │ ← separately trained          No reward model needed!
├──────────────┤                               No value function needed!
│ Value Function│ ← for PPO baseline           No online sampling needed!
└──────────────┘
```

**What DPO gains:**
- No reward model training (saves compute and avoids reward model errors)
- No online sampling during training (just forward passes on static preference data)
- No PPO instability (simple supervised loss)
- Half the memory footprint (2 models instead of 4)

**What DPO assumes:**
- The Bradley-Terry preference model is correct
- The preference data is representative of the true human preference distribution
- The reference model is a reasonable starting point

### DPO Gradient Analysis

The gradient of the DPO loss with respect to $\theta$ has an intuitive interpretation:

$$\nabla_\theta \mathcal{L}_{\text{DPO}} = -\beta \, \mathbb{E} \left[ \underbrace{\sigma(-\hat{r}_\theta(x, y_w) + \hat{r}_\theta(x, y_l))}_{\text{weight: higher when model is wrong}} \left( \underbrace{\nabla_\theta \log \pi_\theta(y_w|x)}_{\text{increase chosen prob}} - \underbrace{\nabla_\theta \log \pi_\theta(y_l|x)}_{\text{decrease rejected prob}} \right) \right]$$

The sigmoid weight is large when the model assigns higher implicit reward to the *rejected* response — exactly when the model is making mistakes. As training progresses and the model learns the correct preferences, the weight shrinks and updates become smaller.

🎯 **Interview tip:** DPO's gradient has a natural curriculum effect — it focuses learning on examples where the model is most confused. This is similar to hard example mining in contrastive learning.

---

## Math Derivation: KTO (Kahneman-Tversky Optimization)

DPO requires **paired** preference data — for each prompt, you need both a chosen and a rejected response. KTO relaxes this requirement using insights from **prospect theory** (Kahneman & Tversky, 1979).

### The Problem with Paired Data

Collecting paired preferences is expensive:
1. Generate two responses per prompt
2. Have a human compare them and pick the better one
3. Repeat thousands of times

In practice, it's much easier to collect **unpaired** feedback: individual responses labeled as "good" (👍) or "bad" (👎), without requiring direct comparison.

### Prospect Theory Connection

Kahneman and Tversky showed that humans evaluate outcomes relative to a **reference point**, and that losses loom larger than gains. KTO applies this insight to alignment:

- A "good" response should have higher implicit reward than the reference point
- A "bad" response should have lower implicit reward than the reference point
- The penalty for being wrong about bad responses should be larger (loss aversion)

### The KTO Loss

For a desirable response $y_d$ (👍) and an undesirable response $y_u$ (👎):

$$\mathcal{L}_{\text{KTO}} = \mathbb{E}_{x, y_d}\left[w_d \cdot \left(1 - \sigma\left(\beta \log \frac{\pi_\theta(y_d|x)}{\pi_{\text{ref}}(y_d|x)} - z_{\text{ref}}\right)\right)\right] + \mathbb{E}_{x, y_u}\left[w_u \cdot \left(1 - \sigma\left(z_{\text{ref}} - \beta \log \frac{\pi_\theta(y_u|x)}{\pi_{\text{ref}}(y_u|x)}\right)\right)\right]$$

where:
- $z_{\text{ref}} = \mathbb{E}_{x'}\left[\beta \, \text{KL}\big(\pi_\theta(\cdot|x') \| \pi_{\text{ref}}(\cdot|x')\big)\right]$ is the reference point (average KL divergence)
- $w_d, w_u$ are weights reflecting loss aversion (typically $w_u > w_d$, penalizing bad outputs more)

### KTO vs DPO: When to Use Each

| Aspect | DPO | KTO |
|--------|-----|-----|
| **Data format** | Paired (chosen vs rejected) | Unpaired (good 👍 or bad 👎) |
| **Data collection** | Expensive (need comparisons) | Cheaper (binary labels) |
| **Theoretical basis** | Bradley-Terry model | Prospect theory |
| **Loss aversion** | No | Yes ($w_u > w_d$) |
| **Performance** | Strong with good pairs | Competitive, especially with noisy data |

💡 **Key insight:** KTO shows that you don't need paired comparisons to do alignment — binary feedback is sufficient. This dramatically reduces data collection costs.

---

## Math Derivation: GRPO (Group Relative Policy Optimization)

GRPO (DeepSeek, 2025) takes a fundamentally different approach: instead of learning from human preferences, it uses **self-generated responses** and **group-relative normalization** to create its own training signal.

### The Core Idea

For each prompt $x$:
1. **Sample a group** of $N$ responses: $\{y_1, y_2, \ldots, y_N\} \sim \pi_\theta(\cdot|x)$
2. **Score each response** using a reward function $r(x, y_i)$ (can be a rule-based verifier, not necessarily a learned reward model)
3. **Normalize rewards within the group** to mean ≈ 0, std ≈ 1
4. **Update the policy** using policy gradient weighted by normalized rewards

### Step 1: Group Sampling

Generate $N$ responses (typically $N = 8$ to $64$) from the current policy:

$$y_i \sim \pi_\theta(\cdot|x), \quad i = 1, \ldots, N$$

### Step 2: Reward Computation

Score each response. Crucially, the reward can be a **rule-based verifier** (e.g., "is the math answer correct?") rather than a learned reward model:

$$r_i = r(x, y_i), \quad i = 1, \ldots, N$$

### Step 3: Group Normalization

Normalize rewards within the group to remove prompt-level difficulty bias:

$$\hat{r}_i = \frac{r_i - \text{mean}(\{r_1, \ldots, r_N\})}{\text{std}(\{r_1, \ldots, r_N\}) + \epsilon}$$

After normalization: $\text{mean}(\hat{r}) \approx 0$ and $\text{std}(\hat{r}) \approx 1$.

💡 **Why normalize?** Different prompts have different difficulty levels. A reward of 0.7 might be excellent for a hard math problem but mediocre for a simple question. Group normalization makes rewards comparable across prompts by measuring each response relative to its peers.

### Step 4: Policy Gradient with KL Penalty

The GRPO loss combines the normalized reward signal with a KL penalty:

$$\boxed{\mathcal{L}_{\text{GRPO}} = -\frac{1}{N} \sum_{i=1}^{N} \left[ \hat{r}_i \cdot \min\left(\frac{\pi_\theta(y_i|x)}{\pi_{\text{old}}(y_i|x)}, \text{clip}(1-\epsilon, 1+\epsilon)\right) - \beta \cdot \text{KL}\big(\pi_\theta \| \pi_{\text{ref}}\big) \right]}$$

where:
- $\hat{r}_i$ is the group-normalized reward for response $i$
- The $\min(\cdot, \text{clip}(\cdot))$ is the PPO-style clipped ratio (prevents too-large updates)
- $\beta \cdot \text{KL}$ prevents drift from the reference policy
- $\pi_{\text{old}}$ is the policy that generated the samples (for importance weighting)

### Simplified View: The Intuition

Stripping away the clipping and KL terms, the core gradient is:

$$\nabla_\theta \mathcal{L}_{\text{GRPO}} \propto -\frac{1}{N} \sum_{i=1}^{N} \hat{r}_i \cdot \nabla_\theta \log \pi_\theta(y_i|x)$$

This is a **REINFORCE-style policy gradient** where:
- Responses with $\hat{r}_i > 0$ (better than group average) get their probability **increased**
- Responses with $\hat{r}_i < 0$ (worse than group average) get their probability **decreased**
- The magnitude of the update is proportional to how far from average the response is

⚡ **Why GRPO is efficient:** No reward model training needed. No paired preference data needed. The group normalization acts as a built-in baseline, reducing variance of the policy gradient estimate.

### GRPO: Key Properties

**Group size matters.** Larger groups give better reward normalization (more stable gradients) but cost more compute per update:

| Group Size $N$ | Normalization Quality | Compute Cost | Typical Use |
|:-:|:-:|:-:|:-:|
| 2 | Poor (high variance) | Low | Not recommended |
| 8 | Reasonable | Moderate | Quick experiments |
| 16–32 | Good | High | Standard training |
| 64 | Excellent | Very high | DeepSeek-R1 |

**No critic/value model.** Unlike PPO-based RLHF, GRPO doesn't need a value function to estimate baselines. The group mean *is* the baseline. This eliminates one of the four models from the RLHF pipeline.

**Works with verifiable rewards.** GRPO shines when you have a **rule-based reward** (math correctness, code execution, format compliance) rather than a learned reward model. This is why DeepSeek used it for reasoning — you can check if a math answer is correct without a neural network.

🎯 **Interview tip:** GRPO's main innovation is replacing the learned value function baseline with group-relative normalization. This is simpler, more stable, and works especially well for tasks with verifiable rewards (math, code, logic).

---

## The Full Progression: SFT → RLHF → DPO → KTO → GRPO

Each method in the alignment lineage solves a problem introduced by its predecessor:

```
SFT                    RLHF                   DPO                    KTO                    GRPO
────────────────────   ────────────────────   ────────────────────   ────────────────────   ────────────────────
Problem: Base model    Problem: SFT can't     Problem: RLHF needs   Problem: DPO needs     Problem: All above
can't follow           distinguish quality    4 models + PPO        paired preferences     need human labels
instructions           among valid responses  (unstable, expensive)  (expensive to collect) (expensive, noisy)

Solution: Fine-tune    Solution: Train        Solution: Reparame-   Solution: Use          Solution: Self-play
on (instruction,       reward model, use      terize reward as      unpaired binary        with group-normalized
response) pairs        PPO to optimize        policy log-ratio,     feedback + prospect    rewards from
                       reward - β·KL          eliminate reward       theory loss            verifiable signals
                                              model entirely         aversion

Data: Demonstrations   Data: Preference       Data: Preference      Data: Binary           Data: Prompts only
(x, y)                 pairs (x, y_w, y_l)   pairs (x, y_w, y_l)  labels (x, y, 👍/👎)   (self-generated y's)

Models: 1              Models: 4              Models: 2             Models: 2              Models: 2
(policy)               (π, π_ref, r, V)      (π, π_ref)           (π, π_ref)            (π, π_ref)

Loss:                  Loss:                  Loss:                 Loss:                  Loss:
CE on demos            PPO + KL penalty       -log σ(β·Δlog-ratio) Prospect theory        Policy gradient with
                                                                    based                  group normalization
```

💡 **The trend is clear:** Each generation reduces the amount of human supervision needed, from demonstrations (SFT) to paired preferences (RLHF/DPO) to binary labels (KTO) to no human labels at all (GRPO with verifiable rewards).

---

## Summary of Key Equations

For quick reference, here are the core equations side by side:

| Method | Loss Function |
|--------|--------------|
| **SFT** | $\mathcal{L} = -\sum_t \log \pi_\theta(y_t \mid x, y_{<t})$ |
| **RLHF** | $\max \; \mathbb{E}[r(x,y)] - \beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})$ |
| **DPO** | $\mathcal{L} = -\log \sigma\!\left(\beta\!\left(\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right)$ |
| **KTO** | Prospect-theory weighted loss on unpaired good/bad examples |
| **GRPO** | $\mathcal{L} = -\frac{1}{N}\sum_i \hat{r}_i \cdot \log \pi_\theta(y_i|x) - \beta \cdot \text{KL}$ |

where $\hat{r}_i = \frac{r_i - \mu_{\text{group}}}{\sigma_{\text{group}}}$ is the group-normalized reward.

🎯 **Interview tip:** Be able to derive DPO from RLHF in 3 steps: (1) write the optimal RLHF policy, (2) solve for reward in terms of policy, (3) substitute into Bradley-Terry and watch $Z(x)$ cancel.

---

## Setup

### 🧠 The Big Picture

Alignment is like training a new employee — RLHF is like having a manager (reward model) watch every action and give feedback, DPO is like showing pairs of examples ('this response is better than that one') and letting the employee figure out the pattern, and GRPO is like having a group discussion where the best ideas rise to the top.

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

### 📖 Key Terms

Before we dive in, here are some terms you'll encounter:

- **Loss Function**: a mathematical formula that measures how wrong the model output is (the goal of training is to minimize this)
- **Embedding**: a dense vector representation of a token — a list of numbers that captures the token meaning
- **Transformer**: the neural network architecture that processes sequences using self-attention
- **Encoder**: a model component that reads and understands input text (used in BERT-style models)
- **Decoder**: a model component that generates text one token at a time (used in GPT-style models)

### 📦 Dependencies

The following cell imports the libraries we need. Just run it — we will explain each one as we use it.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

mx.random.seed(42)
print(f"MLX version: {mx.__version__}")
print("✅ Ready for Alignment: RLHF, DPO, and GRPO!")

---

## Implementation: Reward Model in MLX

Following the pedagogical pattern: *math derivation → step-by-step code → visualization → comparison.*

We derived the reward model math above — now let's build one. The reward model takes a (prompt, response) token sequence and outputs a scalar quality score:

$$r_\phi(x, y) = W_r^\top \cdot h_{\text{last}}(x, y) + b_r$$

💡 **Architecture recap:** We use a small transformer as the base model, replace the LM head with a linear projection to a scalar, and extract the hidden state at the **last token position** (which has seen the entire sequence via causal attention).

🎯 **Interview tip:** In production RLHF, the reward model is typically initialized from the SFT model checkpoint. Here we build a small one from scratch to show the architecture clearly.

⚠️ **Note:** Our model is intentionally tiny (d_model=64, 2 layers) for demonstration. Real reward models use billions of parameters.

In [ ]:
from utils.alignment import RewardModelConfig, RewardModel

# Create a small reward model
config = RewardModelConfig(
    d_model=64,
    n_heads=4,
    n_layers=2,
    vocab_size=256,
    max_seq_len=128,
)
model = RewardModel(config)  # shape: function output

print(f"Reward Model Config:")
print(f"  d_model:     {config.d_model}")
print(f"  n_heads:     {config.n_heads}")
print(f"  n_layers:    {config.n_layers}")
print(f"  vocab_size:  {config.vocab_size}")
print(f"  max_seq_len: {config.max_seq_len}")
print(f"  head_dim:    {config.d_model // config.n_heads}")

The next cell continues the implementation:

**Forward pass: (prompt, response) → scalar reward**

In [ ]:
# Forward pass: (prompt, response) → scalar reward
# Simulate two (prompt, response) sequences as token IDs
batch_size, seq_len = 2, 20
input_ids = mx.random.randint(0, config.vocab_size, shape=(batch_size, seq_len))

reward = model(input_ids)  # shape: function output
mx.eval(reward)

print(f"Input shape:  {input_ids.shape}  — [batch, seq]")
print(f"Reward shape: {reward.shape}  — [batch, 1]")
print(f"Reward values: {reward.tolist()}")
print()
print("⚡ Each (prompt, response) pair maps to a single scalar reward.")
print("   The reward head projects the last-token hidden state → ℝ¹.")

### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [ ]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])  # shape: see output
    result = mx.sum(test)  # shape: see output
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

The next cell continues the implementation:

**Convenience method: compute_reward returns [batch] instead of [batch, 1]**

In [ ]:
# Convenience method: compute_reward returns [batch] instead of [batch, 1]
scalar_rewards = model.compute_reward(input_ids)
mx.eval(scalar_rewards)

print(f"compute_reward shape: {scalar_rewards.shape}  — [batch]")
print(f"Rewards: {scalar_rewards.tolist()}")
print()
print("💡 In RLHF, we train this model on preference pairs (chosen vs rejected)")
print("   using the Bradley-Terry loss: L = -log σ(r(x, y_w) - r(x, y_l))")

⚡ **What just happened:**
1. Token IDs → embedding + sinusoidal positional encoding → `[batch, seq, 64]`
2. Two transformer blocks (causal self-attention + SiLU-gated FFN) → `[batch, seq, 64]`
3. Extract hidden state at the **last token** position → `[batch, 64]`
4. Linear reward head projects to scalar → `[batch, 1]`

The last-token position is key: because attention is causal, the final token's hidden state has attended to the entire (prompt, response) sequence — it's the natural summary vector.

🎯 **Interview tip:** Be ready to explain why we use the *last* token rather than mean-pooling. In causal (decoder-only) models, only the last position has seen all prior tokens. Encoder models (like BERT) would use `[CLS]` or mean-pooling instead.

---

## Implementation: DPO Training in MLX

Now let's implement the DPO loss and training loop from scratch. Recall the key equation:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \log \frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\right)\right)\right]$$

We need three components:
1. **SimpleLM** — a small language model that outputs logits (serves as both policy and reference)
2. **DPOTrainer** — computes log-probs, DPO loss, and runs the training loop
3. **NaN recovery** — if training diverges, reload checkpoint and reduce LR by 10×

💡 **Key insight:** DPO only needs 2 models in memory (policy + reference), compared to RLHF's 4. The reference model stays completely frozen — we never compute gradients through it.

⚠️ **Note:** We use `mx.stop_gradient()` on reference model outputs and `model.freeze()` to ensure no gradient updates leak through.

In [ ]:
# 📊 What does preference data ACTUALLY look like?
# Before we train DPO, let's see what the training data looks like in practice.

print("=" * 70)
print("EXAMPLE PREFERENCE DATA (what humans label)")
print("=" * 70)

examples = [
    {
        "prompt": "Explain quantum computing to a 5-year-old.",
        "chosen": "Imagine you have a magic coin that can be heads AND tails at the same time! Quantum computers use tiny things like that magic coin to solve puzzles really fast.",
        "rejected": "Quantum computing leverages quantum mechanical phenomena such as superposition and entanglement to perform computations on qubits rather than classical bits.",
    },
    {
        "prompt": "Is it safe to eat raw chicken?",
        "chosen": "No, eating raw chicken is dangerous. It can contain Salmonella bacteria that cause food poisoning. Always cook chicken to 165°F (74°C).",
        "rejected": "Some cultures do eat raw chicken dishes. It's a personal choice.",
    },
]

for i, ex in enumerate(examples):
    print(f"\n--- Example {i+1} ---")
    print(f"PROMPT:     {ex['prompt']}")
    print(f"CHOSEN  ✅: {ex['chosen'][:100]}...")
    print(f"REJECTED ❌: {ex['rejected'][:100]}...")

print("\n" + "=" * 70)
print("In DPO, we DON'T train a reward model on this data.")
print("Instead, we directly optimize: P(chosen > rejected) should be HIGH.")
print("The model learns to assign higher probability to chosen-style responses.")
print("=" * 70)

In [ ]:
from utils.alignment import DPOConfig, DPOTrainer, SimpleLM, RewardModelConfig

# --- Create policy and reference models (same architecture) ---
lm_config = RewardModelConfig(
    d_model=64, n_heads=4, n_layers=2, vocab_size=256, max_seq_len=128
)

mx.random.seed(42)
policy_model = SimpleLM(lm_config)
reference_model = SimpleLM(lm_config)

# DPO config: β controls the KL penalty strength
dpo_config = DPOConfig(beta=0.1, learning_rate=1e-4)

print(f"DPO Config:")
print(f"  β (KL penalty):    {dpo_config.beta}")
print(f"  Learning rate:     {dpo_config.learning_rate}")
print(f"  Max length:        {dpo_config.max_length}")
print(f"\nPolicy model:    SimpleLM (d_model=64, 2 layers, vocab=256)")
print(f"Reference model: SimpleLM (same architecture, frozen)")

The next cell continues the implementation:

**--- Create dummy preference data ---**

In [ ]:
# --- Create dummy preference data ---
# In real DPO, these would be tokenized (prompt + chosen) and (prompt + rejected) pairs
# Here we simulate with random token IDs to demonstrate the mechanics
mx.random.seed(123)
batch_size, seq_len = 4, 16

chosen_ids = mx.random.randint(0, lm_config.vocab_size, shape=(batch_size, seq_len))
rejected_ids = mx.random.randint(0, lm_config.vocab_size, shape=(batch_size, seq_len))

print(f"Preference batch:")
print(f"  Chosen shape:   {chosen_ids.shape}  — [batch, seq]")
print(f"  Rejected shape: {rejected_ids.shape}  — [batch, seq]")
print(f"  Batch size: {batch_size}, Sequence length: {seq_len}")

The next cell continues the implementation:

**--- Demonstrate log-prob computation ---**

In [ ]:
# --- Demonstrate log-prob computation ---
log_probs_chosen = DPOTrainer.compute_log_probs(policy_model, chosen_ids)
log_probs_rejected = DPOTrainer.compute_log_probs(policy_model, rejected_ids)
mx.eval(log_probs_chosen, log_probs_rejected)

print("Per-sequence log probabilities under policy model:")
print(f"  Chosen:   {log_probs_chosen.tolist()}")
print(f"  Rejected: {log_probs_rejected.tolist()}")
print()
print("💡 These are sums of log P(next_token | previous_tokens) over the sequence.")
print("   More negative = lower probability under the model.")

The next cell continues the implementation:

**--- Create trainer and run DPO training ---**

In [ ]:
# --- Create trainer and run DPO training ---
trainer = DPOTrainer(dpo_config, policy_model, reference_model)

# Capture reference params before training
import mlx.nn as nn
ref_params_before = [
    (k, v.tolist()) for k, v in nn.utils.tree_flatten(reference_model.parameters())
]

# Train for several steps
num_steps = 10
losses = []
print("DPO Training Loop:")
print(f"{'Step':>4}  {'Loss':>10}")
print("-" * 18)
for step in range(num_steps):
    loss_val = trainer.train_step(chosen_ids, rejected_ids)
    losses.append(loss_val)
    print(f"{step+1:>4}  {loss_val:>10.6f}")

print(f"\n⚡ Loss decreased from {losses[0]:.4f} → {losses[-1]:.4f}")
print("   The policy is learning to prefer chosen over rejected responses.")

The next cell continues the implementation:

**--- Verify reference model params didn't change ---**

In [ ]:
# --- Verify reference model params didn't change ---
ref_params_after = [
    (k, v.tolist()) for k, v in nn.utils.tree_flatten(reference_model.parameters())
]

all_frozen = all(
    b_val == a_val
    for (_, b_val), (_, a_val) in zip(ref_params_before, ref_params_after)
)

print(f"Reference model frozen: {'✅ YES' if all_frozen else '❌ NO'}")
print(f"Policy model updated:   ✅ YES (loss decreased over {num_steps} steps)")
print()
print("🎯 Interview tip: In DPO, the reference model acts as an anchor.")
print("   It prevents the policy from drifting too far (reward hacking).")
print("   The β parameter controls this tradeoff — higher β = more conservative.")

The next cell continues the implementation:

**--- Visualize training loss ---**

In [ ]:
# --- Visualize training loss ---
plt.figure(figsize=(8, 4))
plt.plot(range(1, num_steps + 1), losses, 'b-o', linewidth=2, markersize=5)
plt.xlabel('Training Step')
plt.ylabel('DPO Loss')
plt.title('DPO Training: Loss Decreasing Over Steps')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 The loss decreases because the policy learns to assign higher")
print("   log-probability to chosen responses relative to rejected ones.")
print("   At convergence, the implicit reward margin (chosen - rejected) is maximized.")

---

## Implementation: GRPO Training in MLX

Now let's implement GRPO — the method behind DeepSeek-R1. Recall the core idea:

1. **Sample a group** of N responses from the policy
2. **Score** each with a reward function (can be rule-based — no learned reward model needed)
3. **Normalize** rewards within the group: $\hat{r}_i = \frac{r_i - \mu}{\sigma + \epsilon}$
4. **Update** via policy gradient: $\mathcal{L} = -\frac{1}{N}\sum_i \hat{r}_i \cdot \log \pi_\theta(y_i|x) + \beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})$

💡 **Key insight:** The group mean acts as a built-in baseline, replacing the learned value function in PPO-based RLHF. Responses better than the group average get upweighted; worse ones get downweighted.

⚡ **Why this matters:** GRPO works with *verifiable* rewards (math correctness, code execution) — no expensive human preference data needed.

⚠️ **Note:** Our demo uses a simple rule-based reward (sequence diversity score) to show the mechanics. Real GRPO uses task-specific verifiers.

In [ ]:
from utils.alignment import GRPOConfig, GRPOTrainer, SimpleLM, RewardModelConfig

# --- Create policy and reference models ---
mx.random.seed(42)
grpo_lm_config = RewardModelConfig(
    d_model=64, n_heads=4, n_layers=2, vocab_size=256, max_seq_len=128
)
policy = SimpleLM(grpo_lm_config)
reference = SimpleLM(grpo_lm_config)

# --- Define a simple rule-based reward function ---
# Reward = number of unique tokens / sequence length (diversity score)
def diversity_reward(input_ids: mx.array) -> mx.array:
    """Rule-based reward: fraction of unique tokens per sequence."""
    n, seq = input_ids.shape
    rewards = []
    for i in range(n):
        row = input_ids[i]
        unique_count = len(set(row.tolist()))
        rewards.append(unique_count / seq)
    return mx.array(rewards)

# --- Create GRPO trainer ---
grpo_config = GRPOConfig(group_size=8, beta=0.1, learning_rate=1e-4, max_gen_len=16)
grpo_trainer = GRPOTrainer(grpo_config, policy, reference, diversity_reward)

print("GRPO Config:")
print(f"  Group size:    {grpo_config.group_size}")
print(f"  β (KL penalty): {grpo_config.beta}")
print(f"  Learning rate: {grpo_config.learning_rate}")
print(f"  Max gen len:   {grpo_config.max_gen_len}")
print(f"\nReward function: token diversity (unique / total)")

The next cell continues the implementation:

**--- Step 1: Sample a group of responses ---**

In [ ]:
# --- Step 1: Sample a group of responses ---
mx.random.seed(99)
prompt = mx.array([[1, 2, 3, 4, 5]])  # Simple 5-token prompt
print(f"Prompt shape: {prompt.shape}  — [1, prompt_len]")
print(f"Prompt tokens: {prompt.tolist()[0]}")
print()

group = grpo_trainer.sample_group(prompt, n=8)
print(f"Group shape:  {group.shape}  — [n, prompt_len + max_gen_len]")
print(f"Each response = {prompt.shape[1]} prompt tokens + {grpo_config.max_gen_len} generated tokens")
print()
for i in range(min(4, group.shape[0])):
    print(f"  Response {i}: {group[i].tolist()[:10]}... (first 10 tokens)")

The next cell continues the implementation:

**--- Step 2: Compute raw rewards ---**

In [ ]:
# --- Step 2: Compute raw rewards ---
raw_rewards = grpo_trainer.compute_group_rewards(group)
mx.eval(raw_rewards)

print("Raw rewards (diversity scores):")
for i, r in enumerate(raw_rewards.tolist()):
    print(f"  Response {i}: {r:.4f}")
print(f"\nMean: {mx.mean(raw_rewards).item():.4f}")
print(f"Std:  {mx.sqrt(mx.mean((raw_rewards - mx.mean(raw_rewards))**2)).item():.4f}")

The next cell continues the implementation:

**--- Step 3: Normalize rewards within the group ---**

In [ ]:
# --- Step 3: Normalize rewards within the group ---
norm_rewards = GRPOTrainer.normalize_rewards(raw_rewards)
mx.eval(norm_rewards)

print("Normalized rewards (mean ≈ 0, std ≈ 1):")
for i, r in enumerate(norm_rewards.tolist()):
    marker = "↑" if r > 0 else "↓"
    print(f"  Response {i}: {r:+.4f}  {marker} {'better' if r > 0 else 'worse'} than group avg")

norm_mean = mx.mean(norm_rewards).item()
norm_std = mx.sqrt(mx.mean((norm_rewards - mx.mean(norm_rewards))**2)).item()
print(f"\n✅ Mean: {norm_mean:.6f}  (target ≈ 0)")
print(f"✅ Std:  {norm_std:.6f}  (target ≈ 1)")
print()
print("💡 Group normalization removes prompt-level difficulty bias.")
print("   A reward of 0.7 might be great for a hard prompt but mediocre for an easy one.")
print("   Normalizing within the group makes rewards comparable across prompts.")

The next cell continues the implementation:

**--- Step 4: Compute GRPO loss ---**

In [ ]:
# --- Step 4: Compute GRPO loss ---
loss = grpo_trainer.grpo_loss(prompt, group, norm_rewards)
mx.eval(loss)

print(f"GRPO Loss: {loss.item():.6f}")
print()
print("🎯 The loss combines two terms:")
print("   1. Policy gradient: -mean(r̂_i · log π_θ(y_i))")
print("      → Increases prob of above-average responses, decreases below-average")
print("   2. KL penalty: β · mean(log π_θ - log π_ref)")
print("      → Prevents the policy from drifting too far from the reference")

The next cell continues the implementation:

**--- Visualize: raw vs normalized rewards ---**

In [ ]:
# --- Visualize: raw vs normalized rewards ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

raw_list = raw_rewards.tolist()
norm_list = norm_rewards.tolist()
colors = ['#2ecc71' if r > 0 else '#e74c3c' for r in norm_list]

# Raw rewards
axes[0].bar(range(len(raw_list)), raw_list, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Response Index')
axes[0].set_ylabel('Raw Reward')
axes[0].set_title('Raw Rewards (diversity score)')
axes[0].axhline(y=float(mx.mean(raw_rewards).item()), color='red',
                linestyle='--', label=f'mean={mx.mean(raw_rewards).item():.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Normalized rewards
axes[1].bar(range(len(norm_list)), norm_list, color=colors, alpha=0.8)
axes[1].set_xlabel('Response Index')
axes[1].set_ylabel('Normalized Reward')
axes[1].set_title('Group-Normalized Rewards (mean≈0, std≈1)')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("⚡ Green bars (r̂ > 0): responses better than group average → probability increased")
print("   Red bars (r̂ < 0): responses worse than group average → probability decreased")

---

## Comparison: RLHF vs DPO vs GRPO

Now that we've derived and implemented all three methods, let's put them side by side. The differences matter for choosing the right approach in practice.

| Dimension | RLHF | DPO | GRPO |
|:--|:--|:--|:--|
| **Models in memory** | 4 (policy, reference, reward model, value function) | 2 (policy, reference) | 2 (policy, reference) |
| **Data required** | Paired preferences $(x, y_w, y_l)$ | Paired preferences $(x, y_w, y_l)$ | Prompts only — responses are self-generated |
| **Training loop** | PPO: sample → score → advantage → clip → update | Supervised: forward pass on static pairs | Sample group → score → normalize → policy gradient |
| **Training stability** | ⚠️ PPO is notoriously unstable — sensitive to clipping ratio, learning rate, KL coefficient, value function accuracy | ✅ Stable supervised loss — just binary cross-entropy on log-ratios | ⚡ Moderate — policy gradient has variance, but group normalization helps |
| **Compute cost** | Very high: 4 models + online sampling + PPO iterations | Low: 2 forward passes per batch (policy + reference) | Moderate: N forward passes per prompt (group sampling) |
| **Reward signal** | Learned reward model (expensive to train, can be hacked) | Implicit — reward is the policy log-ratio itself | Flexible — can use rule-based verifiers (math, code) |
| **Empirical quality** | Strong (GPT-4, Claude) | Strong (Llama 3, Zephyr) | Excellent on reasoning (DeepSeek-R1) |
| **Best suited for** | General-purpose alignment with rich preference data | When you have good paired preference data and want simplicity | Tasks with verifiable rewards (math, code, logic) |

### 💡 Key Tradeoffs

- **RLHF** gives the most flexibility (any reward signal) but pays for it with complexity and instability. The reward model can be hacked, and PPO requires careful tuning of at least 5 hyperparameters (clip ratio, KL coefficient, value loss coefficient, learning rate, batch size).

- **DPO** is the simplest to implement — it's literally a supervised loss on preference pairs. The catch: it assumes the Bradley-Terry preference model is correct, and it can't improve beyond the quality of the preference data.

- **GRPO** is the most data-efficient — it only needs prompts, not human labels. But it requires a reliable reward signal (rule-based verifiers work great for math/code, less so for open-ended tasks like creative writing).

### 🎯 Interview tip: When to use which?

```
Have paired preference data?
├── YES → Is training stability critical?
│         ├── YES → DPO (simple, stable, 2 models)
│         └── NO  → RLHF (more flexible, can iterate on reward model)
└── NO  → Do you have verifiable rewards (math, code)?
          ├── YES → GRPO (self-play, no human labels needed)
          └── NO  → Collect preference data first, then DPO
```

⚠️ **The dirty secret:** In practice, most teams start with DPO because it's simple and works well. RLHF is reserved for frontier labs with the engineering resources to stabilize PPO. GRPO is the rising star for reasoning tasks where you can verify correctness automatically.

### Constitutional AI: Alignment Without Human Labelers

**Constitutional AI** (Anthropic, 2022) takes a fundamentally different approach to the labeling bottleneck. Instead of hiring humans to judge model outputs, the model judges *itself* using a set of principles (the "constitution").

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CONSTITUTIONAL AI PIPELINE                       │
│                                                                     │
│  Step 1: GENERATE          Step 2: CRITIQUE         Step 3: REVISE │
│  ┌─────────────────┐      ┌─────────────────┐      ┌─────────────┐│
│  │ Model generates │─────▶│ Model critiques │─────▶│ Model revises││
│  │ a response to   │      │ its own output   │      │ based on the ││
│  │ a prompt        │      │ using principles │      │ critique     ││
│  └─────────────────┘      └─────────────────┘      └─────────────┘│
│                                                                     │
│  "Answer this           "Does this response      "Here's a revised │
│   question..."           violate principle X?"    response that..." │
│                                                                     │
│  Step 4: RLAIF (RL from AI Feedback)                                │
│  ┌─────────────────────────────────────────────────────────────────┐│
│  │ Use the model's own preferences (original vs revised) as       ││
│  │ training signal — no human labelers needed for this step!      ││
│  └─────────────────────────────────────────────────────────────────┘│
└─────────────────────────────────────────────────────────────────────┘
```

#### The Three Key Ideas

1. **Self-critique.** The model generates a response, then is asked to critique it against a set of principles (e.g., "Is this response harmful?", "Does this response contain misinformation?"). The model identifies problems in its own output.

2. **Self-revision.** Based on its critique, the model generates a revised response that addresses the identified issues. This creates (original, revised) pairs — synthetic preference data without any human labeler.

3. **RLAIF (RL from AI Feedback).** The revised responses are treated as "chosen" and the originals as "rejected." This preference data trains a reward model (or is used directly with DPO), replacing human feedback with AI feedback.

#### 💡 Why This Matters

- **Scalability.** Human labeling is expensive ($15–50/hour) and slow. Constitutional AI can generate millions of preference pairs automatically.
- **Consistency.** Human labelers disagree ~30% of the time on preference judgments. A model applying fixed principles is more consistent.
- **Transparency.** The constitution (set of principles) is explicit and auditable. You can see exactly what rules the model is following.

#### ⚠️ Limitations

- The model can only critique as well as it can understand — it may miss subtle harms that a human would catch.
- The constitution must be carefully designed — bad principles lead to bad alignment.
- Self-reinforcing biases: if the model has a blind spot, self-critique won't fix it.

🎯 **Interview tip:** Constitutional AI bridges the gap between RLHF (needs human labels) and GRPO (needs verifiable rewards). It works for open-ended tasks where you can't verify correctness automatically but want to avoid the cost of human labeling. Anthropic's Claude models use this approach extensively.

In [ ]:
# --- Visualization: RLHF vs DPO vs GRPO comparison ---
from utils.alignment import plot_alignment_method_comparison

plot_alignment_method_comparison()

print("⚡ Higher scores = better/easier along that dimension.")
print("   RLHF excels on quality but pays in complexity and compute.")
print("   DPO wins on simplicity and stability — the pragmatic choice.")
print("   GRPO leads on data efficiency — no human labels needed.")

### ⚡ The Alignment Landscape in 2025

The field is converging on a few key insights:

1. **Simpler is often better.** DPO replaced RLHF's 4-model pipeline with a single supervised loss — and matched or exceeded RLHF quality in many benchmarks. The trend toward simplicity continues.

2. **Verifiable rewards unlock self-improvement.** GRPO showed that for tasks where you can check correctness (math, code, logic), you don't need human feedback at all. The model can improve by competing against itself.

3. **AI feedback scales better than human feedback.** Constitutional AI demonstrated that models can critique their own outputs effectively. This is cheaper, faster, and more consistent than human labeling.

4. **The future is hybrid.** Production systems increasingly combine methods: DPO for initial alignment on human preferences, GRPO for reasoning improvement on verifiable tasks, and Constitutional AI for scaling safety without human bottlenecks.

💡 **The meta-lesson:** Each alignment method trades off between data requirements, compute cost, and the assumptions it makes about what "good" means. There's no single best method — the right choice depends on your task, data, and budget.

---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 18: Scaling Laws**, we'll explore predicting model performance from size and data.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?

---

## 📜 History & Alternatives

Alignment is one of the fastest-moving areas in AI. What started as a niche RL problem in 2017 became the defining challenge of modern LLM development by 2023. Here's how we got here.

### Chronological Timeline

| Year | Method | Team | Key Contribution |
|:----:|--------|------|-----------------|
| **2017** | **RLHF concept** | Christiano et al. (OpenAI) | First deep RL from human preferences — trained agents on Atari and MuJoCo using human comparisons instead of hand-crafted reward functions. Proved that human feedback could replace engineered rewards. |
| **2022** | **InstructGPT** | Ouyang et al. (OpenAI) | Applied RLHF to GPT-3 at scale, creating the foundation for ChatGPT. Three-stage pipeline: SFT → reward model → PPO. Showed that a 1.3B RLHF model could be preferred over a 175B base model. |
| **2022** | **Constitutional AI (CAI)** | Bai et al. (Anthropic) | Replaced human labelers with AI self-critique guided by a set of principles ("constitution"). Introduced RLAIF (RL from AI Feedback) — the model critiques and revises its own outputs, then trains on the improved versions. |
| **2023** | **DPO** | Rafailov et al. (Stanford) | Eliminated the reward model entirely via a mathematical reparameterization. Showed the optimal RLHF policy can be recovered with a simple supervised loss on preference pairs. Cut the pipeline from 4 models to 2. |
| **2024** | **KTO** | Ethayarajh et al. (Stanford) | Removed the need for paired preferences by applying Kahneman-Tversky prospect theory. Works with unpaired binary feedback (👍/👎), making data collection dramatically cheaper. Loss aversion weighting penalizes bad outputs more than it rewards good ones. |
| **2024** | **ORPO** | Hong et al. | Odds Ratio Preference Optimization — combined SFT and preference alignment into a single training phase. No reference model needed at all, further simplifying the pipeline. Uses odds ratio of generating chosen vs rejected responses. |
| **2024** | **SimPO** | Meng et al. | Simple Preference Optimization with length-normalized log-probabilities. Addressed DPO's length bias by normalizing rewards by response length. Reference-model-free, making it even simpler than DPO. |
| **2024–2025** | **Online DPO / Iterative DPO** | Multiple teams | Combined online sampling (generating fresh responses from the current policy) with the DPO loss. Bridges the gap between offline DPO and online RLHF — gets DPO's simplicity with RLHF's ability to improve beyond the static dataset. |
| **2025** | **GRPO** | DeepSeek | Group Relative Policy Optimization — replaced the learned value function baseline with group-relative reward normalization. Designed for reasoning tasks with verifiable rewards (math, code). Powers DeepSeek-R1's strong reasoning capabilities. |

### Key Trends

Three clear trajectories emerge from this timeline:

**1. Fewer models, simpler training.**
RLHF required 4 models in memory (policy, reference, reward model, value function). DPO cut it to 2. ORPO and SimPO eliminated the reference model entirely. Each generation removes a component from the pipeline.

**2. Less human supervision.**
The data requirements have steadily decreased: demonstrations (SFT) → paired preferences (RLHF/DPO) → unpaired binary labels (KTO) → AI-generated feedback (CAI) → no human labels at all (GRPO with verifiable rewards).

**3. Task-specific alignment.**
The field is moving away from one-size-fits-all alignment. DPO works well for general helpfulness. GRPO excels at reasoning with verifiable answers. Constitutional AI scales safety without human bottlenecks. Production systems increasingly mix methods.

```
2017          2022              2023         2024                    2025
  │             │                 │            │                       │
  ▼             ▼                 ▼            ▼                       ▼
RLHF ──► InstructGPT ──► DPO ──► KTO ──────────────────────────► GRPO
concept    + CAI                   ORPO, SimPO                  (DeepSeek-R1)
(Christiano) (OpenAI,              Online/Iterative DPO
              Anthropic)  (Stanford)  (Stanford, multiple)       (DeepSeek)
```

### 🎯 Interview Tip

When asked "How would you align a model for task X?", don't default to one method. Show you understand the tradeoffs:

- **General helpfulness with good preference data?** → DPO (simple, stable, well-understood)
- **Limited budget, only thumbs up/down data?** → KTO (works with unpaired feedback)
- **Math/code reasoning with verifiable answers?** → GRPO (self-improvement without human labels)
- **Safety at scale without human bottleneck?** → Constitutional AI / RLAIF
- **Best of both worlds?** → Online DPO or iterative approaches that combine offline simplicity with online exploration

The interviewer wants to see that you can match the method to the problem, not that you memorized one algorithm.